<h2 style="color:#2563eb;">Hafta 5 - Görev 1</h2>

<p>
Geçen haftaki MLP + BatchNorm modelini videodaki gibi küçük adımlara böl
(<strong>X: 3 harf indeksi</strong>, <strong>Y: sıradaki harf</strong>).
</p>

<p>
<strong>loss.backward() ile her ara değişkenin gradient'ini al. </strong> 
</p>



<hr>



In [71]:
import torch
import torch.nn.functional as F


words = open("../hafta3/names.txt", "r").read().splitlines()


chars = sorted(list(set(''.join(words))))
s2i = {s: i + 1 for i, s in enumerate(chars)}
s2i['.'] = 0
i2s = {i: s for s, i in s2i.items()}
vocab_size = len(s2i)


block_size = 3


def build_dataset(words):
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = s2i[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  return torch.tensor(X), torch.tensor(Y)


import random

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])


# 4. HAKEM FONKSİYONU: cmp (Karşılaştırıcı)
# Bizim bulduğumuz türev (dt) ile PyTorch'un bulduğu türevi (t.grad) kıyaslar
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(
      f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff:'
      f' {maxdiff}'
  )

In [72]:
n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
# Katman 1
W1 = (
    torch.randn((block_size * n_embd, n_hidden), generator=g)
    * (5 / 3)
    / ((block_size * n_embd) ** 0.5)
)
b1 = torch.randn(n_hidden, generator=g) * 0.1
# BatchNorm
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1
# Katman 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
  p.requires_grad = True

In [73]:
n = 32
Xb, Yb = Xtr[:n], Ytr[:n]  # Tek bir 32'lik batch

# 1. Embedding
emb = C[Xb]  # (32, 3, 10)
embcat = emb.view(emb.shape[0], -1)  # (32, 30)

# 2. Linear katman 1
hprebn = embcat @ W1 + b1  # (32, 64)

# 3. BatchNorm katmanı (atomik adımlar)
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1 / (n - 1) * (bndiff2).sum(0, keepdim=True)  # Bessel düzeltmesi (n-1)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# 4. Aktivasyon
h = torch.tanh(hpreact)  # (32, 64)

# 5. Linear katman 2
logits = h @ W2 + b2  # (32, 27)

# 6. Cross-Entropy katmanı (atomik adımlar)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch'a tüm ara değişkenlerin gradyanını saklamasını söylüyoruz
for p in parameters:
  p.grad = None
for t in [
    embcat,
    hprebn,
    hpreact,
    h,
    logits,
    logit_maxes,
    norm_logits,
    counts,
    counts_sum,
    counts_sum_inv,
    probs,
    logprobs,
    loss,
    bnraw,
    bnvar,
    bnvar_inv,
    bndiff,
    bndiff2,
    bnmeani,
]:
  t.retain_grad()

# PyTorch referans türevleri hesaplıyor:
loss.backward()

In [74]:
# 1. logprobs ile aynı boyutta (32, 27) sıfırlardan oluşan bir tensör açıyoruz:
dlogprobs = torch.zeros_like(logprobs)

# 2. Sadece doğru karakterlerin olduğu hücrelere -1.0 / n atıyoruz:
dlogprobs[range(n), Yb] = -1.0 / n

# 3. Hakem fonksiyonuyla PyTorch'un hesapladığı ile karşılaştırıyoruz:
cmp('logprobs', dlogprobs, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


In [75]:
dprobs = (1.0 /   probs) * dlogprobs

cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


In [76]:

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)


cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0


In [77]:
dcounts = counts_sum_inv * dprobs

dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv

cmp('counts_sum', dcounts_sum, counts_sum)

counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0


In [78]:

dcounts += torch.ones_like(counts) * dcounts_sum


cmp('counts', dcounts, counts)

counts          | exact: True  | approximate: True  | maxdiff: 0.0


In [79]:

dnorm_logits = counts * dcounts


cmp('norm_logits', dnorm_logits, norm_logits)

norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0


In [80]:

dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)


cmp('logit_maxes', dlogit_maxes, logit_maxes)

logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0


In [81]:

dlogits = dnorm_logits.clone()


dlogits += (
    F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
)


cmp('logits', dlogits, logits)

logits          | exact: True  | approximate: True  | maxdiff: 0.0


In [82]:
dh = dlogits @ W2.T

dW2 = h.T @ dlogits

db2 = dlogits.sum(0)

cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0


In [83]:

dhpreact = (1.0 - h**2) * dh


cmp('hpreact', dhpreact, hpreact)

hpreact         | exact: True  | approximate: True  | maxdiff: 0.0


In [84]:

dbngain = (bnraw * dhpreact).sum(0, keepdim=True)


dbnbias = dhpreact.sum(0, keepdim=True)


dbnraw = bngain * dhpreact


cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)

bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff: 0.0


In [85]:

dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)


dbndiff = bnvar_inv * dbnraw


cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

bnvar_inv       | exact: True  | approximate: True  | maxdiff: 0.0


In [86]:
dbnvar = (-0.5 * (bnvar + 1e-5)**-1.5) * dbnvar_inv

cmp('bnvar', dbnvar, bnvar)

bnvar           | exact: True  | approximate: True  | maxdiff: 0.0


In [87]:
dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar

cmp('bndiff2', dbndiff2, bndiff2)

bndiff2         | exact: True  | approximate: True  | maxdiff: 0.0


In [88]:
dbndiff += (2.0 * bndiff) * dbndiff2

cmp('bndiff', dbndiff, bndiff)

bndiff          | exact: True  | approximate: True  | maxdiff: 0.0


In [89]:
dbnmeani = (-dbndiff).sum(0, keepdim=True)

cmp('bnmeani', dbnmeani, bnmeani)

bnmeani         | exact: True  | approximate: True  | maxdiff: 0.0


In [90]:
dhprebn = dbndiff.clone()

dhprebn += (1.0 / n) * torch.ones_like(hprebn) * dbnmeani

cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: True  | approximate: True  | maxdiff: 0.0


In [91]:
dembcat = dhprebn @ W1.T

dW1 = embcat.T @ dhprebn

db1 = dhprebn.sum(0)

cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

embcat          | exact: True  | approximate: True  | maxdiff: 0.0
W1              | exact: True  | approximate: True  | maxdiff: 0.0
b1              | exact: True  | approximate: True  | maxdiff: 0.0


In [92]:
demb = dembcat.view(emb.shape)

dC = torch.zeros_like(C)

for k in range(Xb.shape[0]):  # 32 batch elemanı
  for j in range(Xb.shape[1]):  # 3 harf (block_size)
    ix = Xb[k, j]
    dC[ix] += demb[k, j]

# Test:
cmp('C', dC, C)

C               | exact: True  | approximate: True  | maxdiff: 0.0


### Görev 2

In [93]:
dlogits = probs.clone()
dlogits[range(n), Yb] -= 1.0
dlogits /= n

cmp('logits', dlogits, logits)

logits          | exact: False | approximate: True  | maxdiff: 6.752088665962219e-09


### BatchNorm Sadeleştirmesi

In [94]:
dhprebn = (
    bngain
    * bnvar_inv
    / n
    * (n * dhpreact - dhpreact.sum(0) - bnraw * (dhpreact * bnraw).sum(0))
)

cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: False | maxdiff: 2.5910441763699055e-05


In [95]:
# Bessel düzeltmeli (n-1) BatchNorm analitik türevi
dhprebn = (
    bngain
    * bnvar_inv
    * (
        dhpreact
        - dhpreact.mean(0)
        - bnraw * (dhpreact * bnraw).sum(0) / (n - 1)
    )
)

# Test:
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09


### Görev 3

In [96]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = (
    torch.randn((block_size * n_embd, n_hidden), generator=g)
    * (5 / 3)
    / ((block_size * n_embd) ** 0.5)
)
b1 = torch.randn(n_hidden, generator=g) * 0.1
bngain = torch.randn((1, n_hidden), generator=g) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden), generator=g) * 0.1
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

for p in parameters:
  p.requires_grad = False


max_steps = 1000
batch_size = 32
n = batch_size

for step in range(max_steps):
  # Mini-batch seçimi
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix]

  
  emb = C[Xb]  
  embcat = emb.view(emb.shape[0], -1) 

  
  hprebn = embcat @ W1 + b1  

  
  bnmean = hprebn.mean(0, keepdim=True)
  bndiff = hprebn - bnmean
  bnvar = (bndiff**2).sum(0, keepdim=True) / (n - 1)  
  bnvar_inv = (bnvar + 1e-5) ** -0.5
  bnraw = bndiff * bnvar_inv
  hpreact = bngain * bnraw + bnbias

  
  h = torch.tanh(hpreact)

  
  logits = h @ W2 + b2  # (32, 27)

  
  logit_maxes = logits.max(1, keepdim=True).values
  norm_logits = logits - logit_maxes
  counts = norm_logits.exp()
  counts_sum_inv = counts.sum(1, keepdim=True) ** -1
  probs = counts * counts_sum_inv
  loss = -probs[range(n), Yb].log().mean()

  
  dlogits = probs.clone()
  dlogits[range(n), Yb] -= 1.0
  dlogits /= n

 
  dh = dlogits @ W2.T
  dW2 = h.T @ dlogits
  db2 = dlogits.sum(0)

  
  dhpreact = (1.0 - h**2) * dh

  
  dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
  dbnbias = dhpreact.sum(0, keepdim=True)

 
  dhprebn = (
      bngain
      * bnvar_inv
      * (
          dhpreact
          - dhpreact.mean(0)
          - bnraw * (dhpreact * bnraw).sum(0) / (n - 1)
      )
  )

 
  dembcat = dhprebn @ W1.T
  dW1 = embcat.T @ dhprebn
  db1 = dhprebn.sum(0)

 
  demb = dembcat.view(emb.shape)
  dC = torch.zeros_like(C)
  for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
      ix_char = Xb[k, j]
      dC[ix_char] += demb[k, j]

  
  grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

  
  lr = 0.1 if step < 500 else 0.01  # Learning rate decay
  for p, grad in zip(parameters, grads):
    p -= lr * grad

  
  if step % 100 == 0 or step == max_steps - 1:
    print(f"Adım {step:4d}/{max_steps} | Kayıp (Loss): {loss.item():.4f}")

Adım    0/1000 | Kayıp (Loss): 3.4696
Adım  100/1000 | Kayıp (Loss): 3.0271
Adım  200/1000 | Kayıp (Loss): 2.8327
Adım  300/1000 | Kayıp (Loss): 2.5382
Adım  400/1000 | Kayıp (Loss): 2.5734
Adım  500/1000 | Kayıp (Loss): 2.2261
Adım  600/1000 | Kayıp (Loss): 2.6100
Adım  700/1000 | Kayıp (Loss): 2.7340
Adım  800/1000 | Kayıp (Loss): 2.4416
Adım  900/1000 | Kayıp (Loss): 2.5415
Adım  999/1000 | Kayıp (Loss): 2.9725


In [ ]:
import time


for p in parameters:
  p.requires_grad = True

t0 = time.time()
for _ in range(100):
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix]

  emb = C[Xb].view(batch_size, -1)
  hprebn = emb @ W1 + b1
  bnmean = hprebn.mean(0, keepdim=True)
  bndiff = hprebn - bnmean
  bnvar = (bndiff**2).sum(0, keepdim=True) / (n - 1)
  bnvar_inv = (bnvar + 1e-5) ** -0.5
  bnraw = bndiff * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Yb)

  for p in parameters:
    p.grad = None
  loss.backward()

t1 = time.time()
pytorch_time = (t1 - t0) / 100 * 1000


for p in parameters:
  p.requires_grad = False

t0 = time.time()
for _ in range(100):
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix]

  
  emb = C[Xb]
  embcat = emb.view(batch_size, -1)
  hprebn = embcat @ W1 + b1
  bnmean = hprebn.mean(0, keepdim=True)
  bndiff = hprebn - bnmean
  bnvar = (bndiff**2).sum(0, keepdim=True) / (n - 1)
  bnvar_inv = (bnvar + 1e-5) ** -0.5
  bnraw = bndiff * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  h = torch.tanh(hpreact)
  logits = h @ W2 + b2

  counts = (logits - logits.max(1, keepdim=True).values).exp()
  probs = counts / counts.sum(1, keepdim=True)

  
  dlogits = probs.clone()
  dlogits[range(n), Yb] -= 1.0
  dlogits /= n

  dh = dlogits @ W2.T
  dW2 = h.T @ dlogits
  db2 = dlogits.sum(0)
  dhpreact = (1.0 - h**2) * dh

  dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
  dbnbias = dhpreact.sum(0, keepdim=True)
  dhprebn = (
      bngain
      * bnvar_inv
      * (
          dhpreact
          - dhpreact.mean(0)
          - bnraw * (dhpreact * bnraw).sum(0) / (n - 1)
      )
  )

  dembcat = dhprebn @ W1.T
  dW1 = embcat.T @ dhprebn
  db1 = dhprebn.sum(0)

  demb = dembcat.view(emb.shape)
  dC = torch.zeros_like(C)
  for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
      dC[Xb[k, j]] += demb[k, j]

t1 = time.time()
manual_time = (t1 - t0) / 100 * 1000

print(f"PyTorch Autograd Adım Süresi : {pytorch_time:.3f} ms")
print(f"Manuel Analitik Adım Süresi   : {manual_time:.3f} ms")
print(f"Hız Farkı                      : {pytorch_time / manual_time:.2f}x")

PyTorch Autograd Adım Süresi : 0.889 ms
Manuel Analitik Adım Süresi   : 2.135 ms
Hız Farkı                      : 0.42x
